# Advanced `frozenset` Problems — With Complete Solutions

This notebook develops advanced, practical mastery of Python's `frozenset`.

## Learning goals

By the end, you should be able to:

- explain why `frozenset` is hashable while `set` is not;
- safely use `frozenset` as a dictionary key or as an element of another set;
- reason about result types when mixing `set` and `frozenset`;
- design canonical, order-independent keys;
- avoid subtle bugs caused by using sets when multiplicity or position matters;
- use `frozenset` in memoization, graphs, hypergraphs, grouping, deduplication, and dynamic programming;
- recursively convert nested mutable structures into hashable structures;
- choose between `tuple` and `frozenset` based on data semantics.

> **Best-practice principle:** use `frozenset` only when **order does not matter** and **duplicates do not matter**.

## 0. Quick reference

In [1]:
# Construction
a = frozenset([1, 2, 2, 3])
b = frozenset({3, 4, 5})

print(a)                        # duplicates removed
print(hash(a))                  # hashable
print(a | b)                    # union
print(a & b)                    # intersection
print(a - b)                    # difference
print(a ^ b)                    # symmetric difference
print(a <= b, a < b)            # subset / proper subset
print(a.isdisjoint({10, 20}))   # no common elements

frozenset({1, 2, 3})
-272375401224217160
frozenset({1, 2, 3, 4, 5})
frozenset({3})
frozenset({1, 2})
frozenset({1, 2, 4, 5})
False False
True


### Important semantic rule

A `frozenset` represents an **unordered collection of unique hashable values**.

So these are equal:

In [2]:
assert frozenset([1, 2, 3]) == frozenset([3, 2, 1])
assert frozenset([1, 1, 2, 3]) == frozenset([1, 2, 3])

print("Order and duplicate multiplicity are intentionally ignored.")

Order and duplicate multiplicity are intentionally ignored.


---

# Problem 1 — Hashability Audit

You are given several candidate objects. Determine which are hashable and explain why.

```python
candidates = [
    frozenset({1, 2, 3}),
    frozenset({(1, 2), (3, 4)}),
    frozenset({frozenset({1, 2}), frozenset({3})}),
    frozenset({("x", (1, 2))}),
]
```

Then test these invalid conceptual cases:

- a `set` itself as a dictionary key;
- a `frozenset` containing a `list`;
- a `frozenset` containing a `set`.

## Solution

In [3]:
def is_hashable(obj):
    try:
        hash(obj)
    except TypeError:
        return False
    return True

candidates = [
    frozenset({1, 2, 3}),
    frozenset({(1, 2), (3, 4)}),
    frozenset({frozenset({1, 2}), frozenset({3})}),
    frozenset({("x", (1, 2))}),
]

for i, obj in enumerate(candidates, start=1):
    print(i, is_hashable(obj), obj)

assert all(is_hashable(x) for x in candidates)

1 True frozenset({1, 2, 3})
2 True frozenset({(1, 2), (3, 4)})
3 True frozenset({frozenset({3}), frozenset({1, 2})})
4 True frozenset({('x', (1, 2))})


In [4]:
# Demonstrate failures safely, without stopping the notebook.

bad_factories = [
    ("set as dict key", lambda: {{1, 2}: "value"}),
    ("frozenset containing list", lambda: frozenset([[1, 2], [3, 4]])),
    ("frozenset containing set", lambda: frozenset([{1, 2}, {3, 4}])),
]

for label, factory in bad_factories:
    try:
        factory()
    except TypeError as exc:
        print(f"{label}: {exc}")

set as dict key: unhashable type: 'set'
frozenset containing list: unhashable type: 'list'
frozenset containing set: unhashable type: 'set'


**Key idea:** `frozenset` is hashable only if every element placed inside it is itself hashable.

---

# Problem 2 — Result-Type Reasoning

Without running the code first, predict the type of every result:

```python
fs = frozenset({1, 2, 3})
s = {3, 4}

fs | s
s | fs
fs & s
s & fs
fs - s
s - fs
fs ^ s
s ^ fs
```

Then verify your prediction.

## Solution

In [5]:
fs = frozenset({1, 2, 3})
s = {3, 4}

expressions = {
    "fs | s": fs | s,
    "s | fs": s | fs,
    "fs & s": fs & s,
    "s & fs": s & fs,
    "fs - s": fs - s,
    "s - fs": s - fs,
    "fs ^ s": fs ^ s,
    "s ^ fs": s ^ fs,
}

for expression, result in expressions.items():
    print(f"{expression:8} -> {result!r:30} type={type(result).__name__}")

fs | s   -> frozenset({1, 2, 3, 4})        type=frozenset
s | fs   -> {1, 2, 3, 4}                   type=set
fs & s   -> frozenset({3})                 type=frozenset
s & fs   -> {3}                            type=set
fs - s   -> frozenset({1, 2})              type=frozenset
s - fs   -> {4}                            type=set
fs ^ s   -> frozenset({1, 2, 4})           type=frozenset
s ^ fs   -> {1, 2, 4}                      type=set


For the binary operators shown above, the **left operand's set type** determines the result type.

This matters when an API promises immutability: put the `frozenset` on the left if you want a frozen result.

---

# Problem 3 — Safe Record Keys

A tempting pattern is:

```python
frozenset({person_name, age})
```

Why is this not a good general-purpose representation of a structured record?

Design a safer `frozenset` key for a person with fields `name` and `age`.

## Solution

In [6]:
def unsafe_person_key(name, age):
    # This discards field identity.
    return frozenset({name, age})

def safe_person_key(name, age):
    # Field names are part of the key.
    return frozenset({
        ("name", name),
        ("age", age),
    })

print(unsafe_person_key("Ada", 36))
print(safe_person_key("Ada", 36))

directory = {
    safe_person_key("Ada", 36): "record-001",
    safe_person_key("Grace", 40): "record-002",
}

assert directory[safe_person_key("Ada", 36)] == "record-001"

frozenset({'Ada', 36})
frozenset({('age', 36), ('name', 'Ada')})


### Best practice

For fixed-position records, a tuple is often even clearer:

```python
("Ada", 36)
```

Use `frozenset` when the **fields themselves form an unordered collection** or when canonical order independence is the point.

---

# Problem 4 — Canonical Undirected Graph Edges

Represent an undirected graph edge so that `(u, v)` and `(v, u)` are treated as the same edge.

Create functions to:

1. build an edge;
2. deduplicate a list of edges;
3. compute all neighbors of a node.

## Solution

In [7]:
def edge(u, v):
    if u == v:
        raise ValueError("Self-loops are not allowed in this example.")
    return frozenset((u, v))

raw_edges = [
    ("A", "B"),
    ("B", "A"),
    ("A", "C"),
    ("C", "D"),
    ("D", "C"),
    ("B", "D"),
]

edges = {edge(u, v) for u, v in raw_edges}
print("Unique edges:", edges)

def neighbors(node, edges):
    result = set()
    for e in edges:
        if node in e:
            # Since every edge has exactly two vertices,
            # subtracting {node} leaves the other endpoint.
            result.update(e - {node})
    return result

assert len(edges) == 4
assert neighbors("A", edges) == {"B", "C"}
assert neighbors("D", edges) == {"B", "C"}

print("Neighbors of A:", neighbors("A", edges))
print("Neighbors of D:", neighbors("D", edges))

Unique edges: {frozenset({'A', 'C'}), frozenset({'A', 'B'}), frozenset({'C', 'D'}), frozenset({'B', 'D'})}
Neighbors of A: {'B', 'C'}
Neighbors of D: {'B', 'C'}


---

# Problem 5 — Count Duplicate Undirected Edges

Given a stream of undirected edges, count how many times each logical edge appears, regardless of endpoint order.

Example:

```python
("A", "B"), ("B", "A"), ("A", "B")
```

must count as the same edge three times.

## Solution

In [8]:
from collections import Counter

stream = [
    ("A", "B"),
    ("B", "A"),
    ("A", "C"),
    ("C", "A"),
    ("A", "B"),
    ("B", "D"),
]

edge_counts = Counter(frozenset((u, v)) for u, v in stream)

for e, count in edge_counts.items():
    print(set(e), "->", count)

assert edge_counts[frozenset({"A", "B"})] == 3
assert edge_counts[frozenset({"A", "C"})] == 2

{'A', 'B'} -> 3
{'A', 'C'} -> 2
{'B', 'D'} -> 1


---

# Problem 6 — Hypergraph Deduplication

A hyperedge can connect any number of vertices, not just two.

Given:

```python
[
    ["A", "B", "C"],
    ["C", "B", "A"],
    ["A", "D"],
    ["D", "A"],
    ["B", "C"],
]
```

deduplicate logically identical hyperedges and compute how many unique hyperedges contain `"A"`.

## Solution

In [9]:
raw_hyperedges = [
    ["A", "B", "C"],
    ["C", "B", "A"],
    ["A", "D"],
    ["D", "A"],
    ["B", "C"],
]

hyperedges = {frozenset(group) for group in raw_hyperedges}

print("Unique hyperedges:")
for h in hyperedges:
    print(" ", h)

containing_a = [h for h in hyperedges if "A" in h]

assert len(hyperedges) == 3
assert len(containing_a) == 2

print("Containing A:", containing_a)

Unique hyperedges:
  frozenset({'B', 'C'})
  frozenset({'A', 'D'})
  frozenset({'C', 'A', 'B'})
Containing A: [frozenset({'A', 'D'}), frozenset({'C', 'A', 'B'})]


---

# Problem 7 — Group Users by Exact Permission Set

Each user has a set of permissions. Users with exactly the same permission set should be grouped together, independent of input order.

Use `frozenset` as the grouping key.

## Solution

In [10]:
from collections import defaultdict

users = {
    "alice": ["read", "write"],
    "bob": ["write", "read"],
    "carol": ["read"],
    "dave": ["read", "admin"],
    "erin": ["admin", "read"],
}

groups = defaultdict(list)

for user, permissions in users.items():
    groups[frozenset(permissions)].append(user)

for permission_set, members in groups.items():
    print(permission_set, "->", members)

assert set(groups[frozenset({"read", "write"})]) == {"alice", "bob"}
assert set(groups[frozenset({"read", "admin"})]) == {"dave", "erin"}

frozenset({'write', 'read'}) -> ['alice', 'bob']
frozenset({'read'}) -> ['carol']
frozenset({'read', 'admin'}) -> ['dave', 'erin']


---

# Problem 8 — Minimal Permission Covers

Suppose each role grants a set of permissions. A role is **redundant** if its permissions are a strict subset of another role's permissions.

Return only the non-redundant roles.

## Solution

In [11]:
roles = {
    "viewer": frozenset({"read"}),
    "editor": frozenset({"read", "write"}),
    "publisher": frozenset({"read", "write", "publish"}),
    "auditor": frozenset({"read", "audit"}),
    "admin": frozenset({"read", "write", "publish", "audit", "delete"}),
}

def non_redundant_roles(role_map):
    result = {}
    items = list(role_map.items())

    for name, permissions in items:
        redundant = any(
            permissions < other_permissions
            for other_name, other_permissions in items
            if other_name != name
        )
        if not redundant:
            result[name] = permissions

    return result

kept = non_redundant_roles(roles)
print(kept)

assert kept == {
    "admin": frozenset({"read", "write", "publish", "audit", "delete"})
}

{'admin': frozenset({'write', 'delete', 'publish', 'read', 'audit'})}


The problem above uses the proper-subset operator `<`.

If two roles have equal permission sets, neither is a **strict** subset of the other. In production code, you may also want to separately detect equivalent roles.

---

# Problem 9 — Find Equivalent Roles

Find role names that grant exactly the same permissions.

## Solution

In [12]:
roles = {
    "content_editor": {"read", "write"},
    "article_editor": {"write", "read"},
    "viewer": {"read"},
    "reader": {"read"},
    "admin": {"read", "write", "delete"},
}

by_permissions = defaultdict(list)

for role, permissions in roles.items():
    by_permissions[frozenset(permissions)].append(role)

equivalent_groups = [
    names
    for names in by_permissions.values()
    if len(names) > 1
]

print(equivalent_groups)

assert any(set(group) == {"content_editor", "article_editor"}
           for group in equivalent_groups)
assert any(set(group) == {"viewer", "reader"}
           for group in equivalent_groups)

[['content_editor', 'article_editor'], ['viewer', 'reader']]


---

# Problem 10 — Basket Deduplication Where Order Does Not Matter

An e-commerce analysis treats a basket as only the set of product IDs purchased. Quantity is intentionally ignored.

Deduplicate baskets:

```python
[
    [101, 102, 103],
    [103, 101, 102],
    [101, 102],
    [101, 101, 102],
]
```

Then explain the semantic consequence of using `frozenset`.

## Solution

In [13]:
baskets = [
    [101, 102, 103],
    [103, 101, 102],
    [101, 102],
    [101, 101, 102],
]

unique_baskets = {frozenset(basket) for basket in baskets}

print(unique_baskets)
assert len(unique_baskets) == 2

# Important:
assert frozenset([101, 101, 102]) == frozenset([101, 102])

{frozenset({101, 102, 103}), frozenset({101, 102})}


The last assertion is the important warning: `frozenset` discards multiplicity.

If quantity matters, use something like:

- `tuple(sorted(items))`, if values are sortable; or
- a frozen representation of a frequency map, such as `frozenset(Counter(items).items())`.

---

# Problem 11 — Preserve Multiplicity with a Frozen Multiset Key

Build a hashable, order-independent key for a bag/multiset where duplicate counts matter.

For example:

```python
[1, 1, 2] != [1, 2, 2]
```

but:

```python
[1, 1, 2] == [2, 1, 1]
```

semantically.

## Solution

In [14]:
def multiset_key(items):
    return frozenset(Counter(items).items())

a = multiset_key([1, 1, 2])
b = multiset_key([2, 1, 1])
c = multiset_key([1, 2, 2])

print("a:", a)
print("b:", b)
print("c:", c)

assert a == b
assert a != c

a: frozenset({(1, 2), (2, 1)})
b: frozenset({(1, 2), (2, 1)})
c: frozenset({(1, 1), (2, 2)})


This is a more precise canonical key than `frozenset(items)` when duplicate counts are meaningful.

---

# Problem 12 — Diagnose a Broken Memoizer

Consider this cache key:

```python
key = frozenset(args)
```

Why can it produce incorrect cache hits for a function such as:

```python
def weighted_sum(a, b):
    return 10*a + b
```

Demonstrate the bug and fix it.

## Solution

In [15]:
def broken_memoizer(fn):
    cache = {}

    def inner(*args):
        key = frozenset(args)  # BUG: loses order and duplicates
        if key not in cache:
            cache[key] = fn(*args)
        return cache[key]

    return inner

@broken_memoizer
def weighted_sum(a, b):
    print("calculating...")
    return 10 * a + b

first = weighted_sum(1, 2)
second = weighted_sum(2, 1)  # Wrongly reuses same cache key!

print(first, second)
assert first == 12
assert second == 12  # demonstrates the bug; correct value should be 21

calculating...
12 12


In [16]:
def positional_memoizer(fn):
    cache = {}

    def inner(*args):
        key = args  # tuple preserves order and duplicates
        if key not in cache:
            cache[key] = fn(*args)
        return cache[key]

    return inner

@positional_memoizer
def weighted_sum_fixed(a, b):
    print("calculating...")
    return 10 * a + b

assert weighted_sum_fixed(1, 2) == 12
assert weighted_sum_fixed(2, 1) == 21

calculating...
calculating...


### Best practice

Do **not** use `frozenset(args)` for ordinary positional arguments.

Use a tuple unless the function is mathematically invariant to argument order **and** duplicate multiplicity is irrelevant.

---

# Problem 13 — Keyword-Order-Independent Memoization

Write a memoizer where:

```python
f(a=1, b=2)
f(b=2, a=1)
```

share the same cache entry.

Preserve positional argument order.

## Solution

In [17]:
from functools import wraps

def memoize_kwargs_order_independent(fn):
    cache = {}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        key = (args, frozenset(kwargs.items()))

        if key not in cache:
            cache[key] = fn(*args, **kwargs)

        return cache[key]

    wrapper.cache = cache
    return wrapper

@memoize_kwargs_order_independent
def combine(prefix, *, a, b):
    print("calculating...")
    return f"{prefix}:{a + b}"

x = combine("sum", a=1, b=2)
y = combine("sum", b=2, a=1)

assert x == y == "sum:3"
assert len(combine.cache) == 1

print("Cache entries:", len(combine.cache))

calculating...
Cache entries: 1


This works only if every value used in the cache key is hashable.

The next problems address nested mutable arguments.

---

# Problem 14 — Recursively Freeze Nested Data

Write a function `freeze(obj)` that converts common nested mutable containers into hashable equivalents:

- `dict` -> `frozenset` of frozen key/value pairs;
- `set` -> `frozenset`;
- `list` -> `tuple` because list order matters;
- `tuple` -> tuple of frozen elements;
- already-hashable scalar values -> unchanged.

The result should be usable as a dictionary key.

## Solution

In [18]:
from collections.abc import Mapping, Set as AbstractSet

def freeze(obj):
    if isinstance(obj, Mapping):
        return frozenset(
            (freeze(key), freeze(value))
            for key, value in obj.items()
        )

    if isinstance(obj, list):
        return tuple(freeze(item) for item in obj)

    if isinstance(obj, tuple):
        return tuple(freeze(item) for item in obj)

    if isinstance(obj, AbstractSet) and not isinstance(obj, (str, bytes)):
        return frozenset(freeze(item) for item in obj)

    # Fail early if the remaining object is not hashable.
    hash(obj)
    return obj

payload = {
    "filters": {
        "regions": {"EU", "US"},
        "years": [2024, 2025],
    },
    "metrics": ["revenue", "margin"],
}

frozen_payload = freeze(payload)

print(frozen_payload)
print("hash =", hash(frozen_payload))

cache = {frozen_payload: "cached-result"}
assert cache[freeze(payload)] == "cached-result"

frozenset({('filters', frozenset({('regions', frozenset({'US', 'EU'})), ('years', (2024, 2025))})), ('metrics', ('revenue', 'margin'))})
hash = -3880950344042789413


### Why is a list converted to a tuple instead of a `frozenset`?

Because list position is normally meaningful. Replacing a list with a `frozenset` would incorrectly erase both ordering and duplicate information.

---

# Problem 15 — Memoize Functions with Nested Mutable Arguments

Use `freeze()` to build a robust memoizer that accepts nested dictionaries, lists, and sets.

## Solution

In [19]:
def memoize_nested(fn):
    cache = {}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        key = (
            freeze(args),
            freeze(kwargs),
        )

        if key not in cache:
            cache[key] = fn(*args, **kwargs)

        return cache[key]

    wrapper.cache = cache
    return wrapper

@memoize_nested
def score(config):
    print("expensive calculation...")
    weights = config["weights"]
    enabled = config["enabled"]
    return sum(weights) * len(enabled)

config1 = {
    "weights": [1, 2, 3],
    "enabled": {"A", "B"},
}

config2 = {
    "enabled": {"B", "A"},
    "weights": [1, 2, 3],
}

assert score(config1) == 12
assert score(config2) == 12
assert len(score.cache) == 1

print("Cache entries:", len(score.cache))

expensive calculation...
Cache entries: 1


Notice the semantics:

- dictionary key order is ignored;
- set order is ignored;
- list order is preserved.

That is usually the right structural interpretation.

---

# Problem 16 — Powerset Generation

Write a function that returns the mathematical powerset of a finite input set.

For `{1, 2, 3}`, the result contains 8 subsets, and each subset should itself be a `frozenset`.

## Solution

In [20]:
from itertools import combinations

def powerset(iterable):
    base = tuple(dict.fromkeys(iterable))  # remove duplicates, preserve stable iteration
    return frozenset(
        frozenset(combo)
        for r in range(len(base) + 1)
        for combo in combinations(base, r)
    )

ps = powerset({1, 2, 3})

print("Number of subsets:", len(ps))
for subset in sorted(ps, key=lambda x: (len(x), tuple(sorted(x)))):
    print(subset)

assert len(ps) == 2 ** 3
assert frozenset() in ps
assert frozenset({1, 2, 3}) in ps

Number of subsets: 8
frozenset()
frozenset({1})
frozenset({2})
frozenset({3})
frozenset({1, 2})
frozenset({1, 3})
frozenset({2, 3})
frozenset({1, 2, 3})


A `frozenset` of `frozenset`s is a natural Python representation of a mathematical set of subsets.

---

# Problem 17 — Unique Pairwise Intersections

Given several sets, compute all **distinct non-empty pairwise intersections**.

## Solution

In [21]:
groups = [
    {"A", "B", "C"},
    {"B", "C", "D"},
    {"C", "D", "E"},
    {"A", "C", "E"},
]

frozen_groups = [frozenset(g) for g in groups]

intersections = {
    frozen_groups[i] & frozen_groups[j]
    for i in range(len(frozen_groups))
    for j in range(i + 1, len(frozen_groups))
}

intersections.discard(frozenset())

print("Distinct intersections:")
for item in sorted(intersections, key=lambda x: (len(x), tuple(sorted(x)))):
    print(item)

assert all(isinstance(x, frozenset) for x in intersections)

Distinct intersections:
frozenset({'C'})
frozenset({'A', 'C'})
frozenset({'B', 'C'})
frozenset({'C', 'D'})
frozenset({'E', 'C'})


---

# Problem 18 — Frequent Itemset Support

Given transaction baskets, compute the support count for each candidate itemset.

A transaction supports a candidate if the candidate is a subset of the transaction.

## Solution

In [22]:
transactions = [
    frozenset({"milk", "bread", "eggs"}),
    frozenset({"milk", "bread"}),
    frozenset({"bread", "butter"}),
    frozenset({"milk", "eggs"}),
    frozenset({"milk", "bread", "butter"}),
]

candidates = {
    frozenset({"milk"}),
    frozenset({"bread"}),
    frozenset({"milk", "bread"}),
    frozenset({"milk", "eggs"}),
    frozenset({"eggs", "butter"}),
}

support = {
    candidate: sum(candidate <= transaction for transaction in transactions)
    for candidate in candidates
}

for candidate, count in sorted(
    support.items(),
    key=lambda pair: (len(pair[0]), sorted(pair[0]))
):
    print(set(candidate), "->", count)

assert support[frozenset({"milk"})] == 4
assert support[frozenset({"milk", "bread"})] == 3
assert support[frozenset({"eggs", "butter"})] == 0

{'bread'} -> 4
{'milk'} -> 4
{'bread', 'milk'} -> 3
{'eggs', 'butter'} -> 0
{'eggs', 'milk'} -> 2


---

# Problem 19 — Apriori-Style Candidate Generation

Suppose you already know these frequent 2-itemsets:

```python
AB, AC, BC, AD
```

Generate unique 3-item candidate sets by unioning pairs of frequent 2-itemsets whose union has size 3.

## Solution

In [23]:
frequent_pairs = {
    frozenset({"A", "B"}),
    frozenset({"A", "C"}),
    frozenset({"B", "C"}),
    frozenset({"A", "D"}),
}

candidate_triples = {
    left | right
    for left in frequent_pairs
    for right in frequent_pairs
    if len(left | right) == 3
}

for candidate in sorted(candidate_triples, key=lambda x: tuple(sorted(x))):
    print(candidate)

assert frozenset({"A", "B", "C"}) in candidate_triples
assert frozenset({"A", "B", "D"}) in candidate_triples
assert frozenset({"A", "C", "D"}) in candidate_triples

frozenset({'B', 'A', 'C'})
frozenset({'A', 'B', 'D'})
frozenset({'A', 'C', 'D'})


---

# Problem 20 — State-Space Search with Immutable States

A state consists of a set of active switches. Toggling a switch returns a new state.

Use `frozenset` states so they can be stored in a `visited` set.

Find all states reachable by toggling switches `"A"`, `"B"`, and `"C"` from the empty state.

## Solution

In [24]:
from collections import deque

switches = frozenset({"A", "B", "C"})

def toggle(state, switch):
    if switch in state:
        return state - {switch}
    return state | {switch}

start = frozenset()
queue = deque([start])
visited = {start}

while queue:
    state = queue.popleft()

    for switch in switches:
        nxt = toggle(state, switch)
        if nxt not in visited:
            visited.add(nxt)
            queue.append(nxt)

print("Reachable states:", len(visited))
for state in sorted(visited, key=lambda s: (len(s), tuple(sorted(s)))):
    print(state)

assert len(visited) == 8

Reachable states: 8
frozenset()
frozenset({'A'})
frozenset({'B'})
frozenset({'C'})
frozenset({'A', 'B'})
frozenset({'A', 'C'})
frozenset({'B', 'C'})
frozenset({'A', 'B', 'C'})


This is a standard pattern in graph search and dynamic programming:

- mutable state representation for updates internally, when useful;
- immutable canonical state representation for cache/visited keys.

---

# Problem 21 — Connected Components from Frozen Undirected Edges

Given an undirected edge set represented by 2-element `frozenset`s, find connected components.

## Solution

In [25]:
edges = {
    frozenset({"A", "B"}),
    frozenset({"B", "C"}),
    frozenset({"D", "E"}),
    frozenset({"F", "G"}),
    frozenset({"G", "H"}),
    frozenset({"F", "H"}),
}

vertices = set().union(*edges)

def build_adjacency(vertices, edges):
    adjacency = {vertex: set() for vertex in vertices}

    for e in edges:
        u, v = tuple(e)
        adjacency[u].add(v)
        adjacency[v].add(u)

    return adjacency

def connected_components(vertices, edges):
    adjacency = build_adjacency(vertices, edges)
    unseen = set(vertices)
    result = set()

    while unseen:
        start = unseen.pop()
        stack = [start]
        component = {start}

        while stack:
            node = stack.pop()
            for neighbor in adjacency[node]:
                if neighbor in unseen:
                    unseen.remove(neighbor)
                    component.add(neighbor)
                    stack.append(neighbor)

        result.add(frozenset(component))

    return frozenset(result)

components = connected_components(vertices, edges)

for component in sorted(components, key=lambda s: tuple(sorted(s))):
    print(component)

assert frozenset({"A", "B", "C"}) in components
assert frozenset({"D", "E"}) in components
assert frozenset({"F", "G", "H"}) in components

frozenset({'C', 'A', 'B'})
frozenset({'E', 'D'})
frozenset({'F', 'G', 'H'})


---

# Problem 22 — Immutable Adjacency Map

Build a fully immutable adjacency representation:

```python
dict[str, frozenset[str]]
```

Then show that individual neighbor collections cannot be mutated.

## Solution

In [26]:
def immutable_adjacency(edges):
    temp = defaultdict(set)

    for u, v in edges:
        temp[u].add(v)
        temp[v].add(u)

    return {
        node: frozenset(neighbors)
        for node, neighbors in temp.items()
    }

adj = immutable_adjacency([
    ("A", "B"),
    ("A", "C"),
    ("B", "D"),
])

print(adj)

assert adj["A"] == frozenset({"B", "C"})

try:
    adj["A"].add("X")
except AttributeError as exc:
    print("Expected immutability:", exc)

{'A': frozenset({'B', 'C'}), 'B': frozenset({'A', 'D'}), 'C': frozenset({'A'}), 'D': frozenset({'B'})}
Expected immutability: 'frozenset' object has no attribute 'add'


The outer dictionary is still mutable. For a fully hashable graph snapshot, freeze the mapping too:

```python
frozenset((node, neighbors) for node, neighbors in adj.items())
```

---

# Problem 23 — Fully Hashable Graph Snapshot

Create a hashable snapshot of an adjacency mapping.

## Solution

In [27]:
graph_snapshot = frozenset(
    (node, neighbors)
    for node, neighbors in adj.items()
)

print(graph_snapshot)
print("Hash:", hash(graph_snapshot))

snapshots = {graph_snapshot}
assert graph_snapshot in snapshots

frozenset({('C', frozenset({'A'})), ('B', frozenset({'A', 'D'})), ('A', frozenset({'B', 'C'})), ('D', frozenset({'B'}))})
Hash: -7326248297062467688


---

# Problem 24 — Detect Disjoint Resource Requests

Each job requests a set of exclusive resources.

Two jobs can run simultaneously if their resource sets are disjoint.

Find every compatible pair.

## Solution

In [28]:
jobs = {
    "job-A": frozenset({"gpu0", "disk1"}),
    "job-B": frozenset({"gpu1"}),
    "job-C": frozenset({"gpu0"}),
    "job-D": frozenset({"disk2"}),
}

job_names = list(jobs)

compatible = []

for i, left in enumerate(job_names):
    for right in job_names[i + 1:]:
        if jobs[left].isdisjoint(jobs[right]):
            compatible.append((left, right))

print("Compatible pairs:")
for pair in compatible:
    print(pair)

assert ("job-A", "job-C") not in compatible
assert ("job-B", "job-D") in compatible

Compatible pairs:
('job-A', 'job-B')
('job-A', 'job-D')
('job-B', 'job-C')
('job-B', 'job-D')
('job-C', 'job-D')


---

# Problem 25 — Immutable Feature Flags as Cache Keys

A compiled artifact depends on a set of feature flags. Flag order does not matter.

Cache compiled results so:

```python
{"fast", "debug"}
{"debug", "fast"}
```

map to the same artifact.

## Solution

In [29]:
compile_cache = {}

def compile_for_flags(flags):
    key = frozenset(flags)

    if key not in compile_cache:
        print("compiling:", sorted(key))
        compile_cache[key] = f"artifact-{len(compile_cache) + 1}"

    return compile_cache[key]

a = compile_for_flags(["fast", "debug"])
b = compile_for_flags(["debug", "fast"])
c = compile_for_flags(["fast"])

assert a == b
assert c != a
assert len(compile_cache) == 2

print(compile_cache)

compiling: ['debug', 'fast']
compiling: ['fast']
{frozenset({'fast', 'debug'}): 'artifact-1', frozenset({'fast'}): 'artifact-2'}


---

# Problem 26 — Maximal Sets

Given a collection of sets, return only those that are not strict subsets of any other set.

Example:

```python
{1}
{1, 2}
{2, 3}
{1, 2, 3}
{4}
```

The maximal sets are `{1, 2, 3}` and `{4}`.

## Solution

In [30]:
family = {
    frozenset({1}),
    frozenset({1, 2}),
    frozenset({2, 3}),
    frozenset({1, 2, 3}),
    frozenset({4}),
}

def maximal_sets(family):
    return {
        candidate
        for candidate in family
        if not any(candidate < other for other in family)
    }

maximal = maximal_sets(family)

print(maximal)
assert maximal == {
    frozenset({1, 2, 3}),
    frozenset({4}),
}

{frozenset({1, 2, 3}), frozenset({4})}


---

# Problem 27 — Minimal Sets

Now return only sets that do not strictly contain any other set in the family.

## Solution

In [31]:
def minimal_sets(family):
    return {
        candidate
        for candidate in family
        if not any(other < candidate for other in family)
    }

minimal = minimal_sets(family)

print(minimal)
assert minimal == {
    frozenset({1}),
    frozenset({2, 3}),
    frozenset({4}),
}

{frozenset({1}), frozenset({2, 3}), frozenset({4})}


---

# Problem 28 — Memoized Recursive Set Partition Search

Determine whether a target set can be exactly covered by some collection of allowed blocks, with no overlaps.

We will memoize by the remaining elements. A `frozenset` is ideal because the remaining state is mathematically a set.

## Solution

In [32]:
from functools import lru_cache

target = frozenset({1, 2, 3, 4})

blocks = (
    frozenset({1, 2}),
    frozenset({3, 4}),
    frozenset({1, 3}),
    frozenset({2, 4}),
    frozenset({1}),
    frozenset({2, 3, 4}),
)

@lru_cache(maxsize=None)
def exact_cover(remaining):
    if not remaining:
        return ()

    pivot = next(iter(remaining))

    for block in blocks:
        if pivot in block and block <= remaining:
            suffix = exact_cover(remaining - block)
            if suffix is not None:
                return (block,) + suffix

    return None

solution = exact_cover(target)

print("Exact cover:", solution)
assert solution is not None
assert frozenset().union(*solution) == target
assert sum(len(block) for block in solution) == len(target)

Exact cover: (frozenset({1, 2}), frozenset({3, 4}))


Here `frozenset` is not just convenient; it expresses the exact semantics of the recursive state.

---

# Problem 29 — Canonicalize Database Query Predicates

Suppose these equality filters are semantically unordered:

```python
country="BG", active=True, tier="pro"
```

Create a canonical hashable query key, while preserving an ordered `sort_by` list.

## Solution

In [33]:
def query_key(filters, sort_by):
    return (
        frozenset(filters.items()),  # unordered predicates
        tuple(sort_by),              # ordered sort priority
    )

k1 = query_key(
    {"country": "BG", "active": True, "tier": "pro"},
    ["created_at", "name"],
)

k2 = query_key(
    {"tier": "pro", "country": "BG", "active": True},
    ["created_at", "name"],
)

k3 = query_key(
    {"tier": "pro", "country": "BG", "active": True},
    ["name", "created_at"],
)

assert k1 == k2
assert k1 != k3

print(k1)

(frozenset({('tier', 'pro'), ('active', True), ('country', 'BG')}), ('created_at', 'name'))


This pattern is a strong example of choosing immutable containers based on semantics:

- unordered unique collection -> `frozenset`;
- ordered sequence -> `tuple`.

---

# Problem 30 — Symmetric Difference for Configuration Drift

Two environments have enabled feature sets.

Find:

1. features enabled in both;
2. features enabled only in production;
3. features enabled only in staging;
4. all drifted features.

## Solution

In [34]:
production = frozenset({
    "payments",
    "search",
    "recommendations",
    "audit",
})

staging = frozenset({
    "payments",
    "search",
    "new-checkout",
    "audit",
})

common = production & staging
only_prod = production - staging
only_staging = staging - production
drift = production ^ staging

print("common:", common)
print("only production:", only_prod)
print("only staging:", only_staging)
print("drift:", drift)

assert common == frozenset({"payments", "search", "audit"})
assert only_prod == frozenset({"recommendations"})
assert only_staging == frozenset({"new-checkout"})
assert drift == frozenset({"recommendations", "new-checkout"})

common: frozenset({'search', 'payments', 'audit'})
only production: frozenset({'recommendations'})
only staging: frozenset({'new-checkout'})
drift: frozenset({'recommendations', 'new-checkout'})


---

# Advanced Challenge 31 — All Minimal Hitting Sets (Small Inputs)

A **hitting set** intersects every set in a family.

For the family:

```python
{"A", "B"}
{"B", "C"}
{"C", "D"}
```

find every inclusion-minimal hitting set by brute force.

This is exponential and intended only for small teaching examples.

## Solution

In [35]:
def minimal_hitting_sets(family):
    family = tuple(frozenset(s) for s in family)

    universe = frozenset().union(*family)
    all_subsets = powerset(universe)

    hitting = {
        candidate
        for candidate in all_subsets
        if all(candidate & group for group in family)
    }

    return frozenset(
        candidate
        for candidate in hitting
        if not any(other < candidate for other in hitting)
    )

family = [
    {"A", "B"},
    {"B", "C"},
    {"C", "D"},
]

hits = minimal_hitting_sets(family)

for h in sorted(hits, key=lambda x: (len(x), tuple(sorted(x)))):
    print(h)

assert all(
    all(h & frozenset(group) for group in family)
    for h in hits
)

frozenset({'A', 'C'})
frozenset({'B', 'C'})
frozenset({'B', 'D'})


---

# Advanced Challenge 32 — Immutable NFA State Sets

In automata algorithms, a DFA state created by subset construction is itself a set of NFA states.

Use `frozenset` to represent DFA states and build a tiny transition table.

## Solution

In [36]:
# Tiny NFA-like transition relation:
# transition[(state, symbol)] -> set of next states

transition = {
    ("q0", "a"): {"q0", "q1"},
    ("q0", "b"): {"q0"},
    ("q1", "b"): {"q2"},
    ("q2", "a"): {"q2"},
}

alphabet = ("a", "b")

def move(state_set, symbol):
    result = set()

    for state in state_set:
        result.update(transition.get((state, symbol), set()))

    return frozenset(result)

start = frozenset({"q0"})

dfa_states = {start}
frontier = [start]
dfa_transition = {}

while frontier:
    current = frontier.pop()

    for symbol in alphabet:
        nxt = move(current, symbol)
        dfa_transition[(current, symbol)] = nxt

        if nxt not in dfa_states:
            dfa_states.add(nxt)
            frontier.append(nxt)

print("DFA states:")
for state in sorted(dfa_states, key=lambda x: (len(x), tuple(sorted(x)))):
    print(state)

print("\nTransitions:")
for (state, symbol), nxt in dfa_transition.items():
    print(state, "--", symbol, "-->", nxt)

assert all(isinstance(state, frozenset) for state in dfa_states)

DFA states:
frozenset({'q0'})
frozenset({'q0', 'q1'})
frozenset({'q0', 'q2'})
frozenset({'q0', 'q1', 'q2'})

Transitions:
frozenset({'q0'}) -- a --> frozenset({'q0', 'q1'})
frozenset({'q0'}) -- b --> frozenset({'q0'})
frozenset({'q0', 'q1'}) -- a --> frozenset({'q0', 'q1'})
frozenset({'q0', 'q1'}) -- b --> frozenset({'q0', 'q2'})
frozenset({'q0', 'q2'}) -- a --> frozenset({'q0', 'q1', 'q2'})
frozenset({'q0', 'q2'}) -- b --> frozenset({'q0'})
frozenset({'q0', 'q1', 'q2'}) -- a --> frozenset({'q0', 'q1', 'q2'})
frozenset({'q0', 'q1', 'q2'}) -- b --> frozenset({'q0', 'q2'})


---

# Advanced Challenge 33 — Immutable Sudoku Candidate Snapshots

Represent the candidates for each cell as `frozenset`s, then produce a new snapshot after eliminating a digit from selected cells.

The goal is to practice **persistent-style updates**: create a new structure instead of mutating candidate sets in place.

## Solution

In [37]:
candidates = {
    (0, 0): frozenset({1, 2, 3}),
    (0, 1): frozenset({2, 3}),
    (1, 0): frozenset({1, 3}),
}

def eliminate(snapshot, cells, digit):
    updated = dict(snapshot)

    for cell in cells:
        if cell in updated:
            updated[cell] = updated[cell] - {digit}

            if not updated[cell]:
                raise ValueError(f"Elimination leaves {cell} with no candidates.")

    return updated

new_candidates = eliminate(candidates, [(0, 0), (0, 1)], 2)

print("Before:", candidates)
print("After: ", new_candidates)

assert candidates[(0, 0)] == frozenset({1, 2, 3})
assert new_candidates[(0, 0)] == frozenset({1, 3})
assert new_candidates[(0, 1)] == frozenset({3})

Before: {(0, 0): frozenset({1, 2, 3}), (0, 1): frozenset({2, 3}), (1, 0): frozenset({1, 3})}
After:  {(0, 0): frozenset({1, 3}), (0, 1): frozenset({3}), (1, 0): frozenset({1, 3})}


---

# Advanced Challenge 34 — Set-of-Sets Dynamic Programming

Suppose each state is a collection of completed task groups, and each task group is itself an unordered set.

Represent the whole state as a `frozenset` of `frozenset`s.

## Solution

In [38]:
group_a = frozenset({"design", "review"})
group_b = frozenset({"implement", "test"})
group_c = frozenset({"deploy"})

state1 = frozenset({group_a, group_b})
state2 = frozenset({group_b, group_a})
state3 = frozenset({group_a, group_c})

assert state1 == state2
assert state1 != state3

memo = {
    state1: "cost=12",
    state3: "cost=7",
}

print(memo[state2])  # same logical state as state1

cost=12


---

# Problem 35 — Performance-Oriented Membership Precomputation

Suppose many repeated queries ask whether a required capability set is contained in a user's capabilities.

Pre-freeze the required sets once, then run many subset checks.

## Solution

In [39]:
required_profiles = {
    "writer": frozenset({"read", "write"}),
    "admin": frozenset({"read", "write", "delete"}),
    "auditor": frozenset({"read", "audit"}),
}

user_capabilities = frozenset({"read", "write", "audit"})

allowed = {
    profile: required <= user_capabilities
    for profile, required in required_profiles.items()
}

print(allowed)

assert allowed == {
    "writer": True,
    "admin": False,
    "auditor": True,
}

{'writer': True, 'admin': False, 'auditor': True}


The main optimization is not that `frozenset` is magically faster than every alternative. It is that the immutable, hashable representation can be safely reused, cached, and shared.

---

# Problem 36 — Equality vs Identity

Show that a mutable `set` and a `frozenset` may compare equal even though:

- they have different types;
- they are different objects.

## Solution

In [40]:
s = {1, 2, 3}
fs = frozenset({3, 2, 1})

print("s == fs:", s == fs)
print("s is fs:", s is fs)
print("types:", type(s).__name__, type(fs).__name__)

assert s == fs
assert s is not fs
assert type(s) is not type(fs)

s == fs: True
s is fs: False
types: set frozenset


---

# Problem 37 — Copy Semantics

Explore these operations:

```python
fs2 = frozenset(fs1)
fs3 = fs1.copy()
```

Check identity.

Then compare with copying a normal set.

## Solution

In [41]:
fs1 = frozenset({1, 2, 3})
fs2 = frozenset(fs1)
fs3 = fs1.copy()

s1 = {1, 2, 3}
s2 = set(s1)
s3 = s1.copy()

print("frozenset(fs1) is fs1:", fs2 is fs1)
print("fs1.copy() is fs1:", fs3 is fs1)
print("set(s1) is s1:", s2 is s1)
print("s1.copy() is s1:", s3 is s1)

assert fs2 is fs1
assert fs3 is fs1
assert s2 is not s1
assert s3 is not s1

frozenset(fs1) is fs1: True
fs1.copy() is fs1: True
set(s1) is s1: False
s1.copy() is s1: False


CPython returns the same object for these shallow-copy-like `frozenset` operations because the object cannot be mutated.

Avoid writing application logic that *requires* identity reuse unless that behavior is explicitly part of the API contract you rely on. Equality is normally the important property.

---

# Problem 38 — Deepcopy Subtlety

Investigate `deepcopy()` on a `frozenset`.

Do not assume it must preserve identity.

## Solution

In [42]:
from copy import deepcopy

fs1 = frozenset({1, 2, 3})
fs2 = deepcopy(fs1)

print("equal:", fs1 == fs2)
print("same object:", fs1 is fs2)

assert fs1 == fs2

equal: True
same object: False


The essential guarantee to depend on here is value equality, not object identity.

---

# Problem 39 — Validate a `frozenset`-Based API

Write a function `normalize_tags(tags)` with these rules:

- accepts any iterable of strings;
- strips whitespace;
- lowercases;
- rejects empty tags after stripping;
- rejects non-string values;
- returns a `frozenset`.

## Solution

In [43]:
def normalize_tags(tags):
    normalized = set()

    for tag in tags:
        if not isinstance(tag, str):
            raise TypeError(f"Tag must be str, got {type(tag).__name__}")

        cleaned = tag.strip().lower()

        if not cleaned:
            raise ValueError("Tags cannot be empty or whitespace-only.")

        normalized.add(cleaned)

    return frozenset(normalized)

tags = normalize_tags([" Python ", "PYTHON", "Data", "data "])

print(tags)
assert tags == frozenset({"python", "data"})

frozenset({'data', 'python'})


This is a common API pattern:

1. validate mutable/untrusted input;
2. normalize it;
3. return an immutable value object.

---

# Problem 40 — Design Decision: `tuple` or `frozenset`?

For each case below, choose the better canonical immutable representation:

1. RGB pixel `(red, green, blue)`
2. mathematical subset of vertices
3. function positional arguments
4. enabled feature flags
5. ordered fallback servers
6. ingredients where duplicates are ignored
7. multiset of dice results where repeated values matter

## Solution

In [44]:
answers = {
    "RGB pixel": "tuple",
    "mathematical subset": "frozenset",
    "function positional arguments": "tuple",
    "feature flags": "frozenset",
    "ordered fallback servers": "tuple",
    "ingredients ignoring duplicates": "frozenset",
    "dice multiset": "frozenset(Counter(values).items())",
}

for case, choice in answers.items():
    print(f"{case:32} -> {choice}")

RGB pixel                        -> tuple
mathematical subset              -> frozenset
function positional arguments    -> tuple
feature flags                    -> frozenset
ordered fallback servers         -> tuple
ingredients ignoring duplicates  -> frozenset
dice multiset                    -> frozenset(Counter(values).items())


## Final best-practice checklist

Use `frozenset` when all of these are true:

- the collection should be immutable;
- element order has no semantic meaning;
- duplicates have no semantic meaning;
- all members can be made hashable;
- you benefit from hashing, dictionary keys, set membership, memoization, or nested sets.

Prefer `tuple` when:

- position matters;
- order matters;
- duplicates matter;
- the object is a fixed record or ordered sequence.

### Common pitfalls to avoid

1. **Do not use `frozenset(args)` for ordinary positional arguments.**  
   It loses position and duplicate multiplicity.

2. **Do not encode structured records as unlabeled values.**  
   Prefer tuples, dataclasses, named tuples, or labeled pairs.

3. **Do not convert lists to frozensets just to make them hashable.**  
   First decide whether list order and duplicates are semantically meaningful.

4. **Do not assume nested mutables magically become hashable.**  
   Recursively freeze them according to their semantics.

5. **Do not rely on set display order.**  
   Sort only for deterministic presentation, not for set semantics.

# Bonus: Compact Self-Test

In [45]:
def run_self_test():
    # Hashability
    assert hash(frozenset({1, 2, 3})) == hash(frozenset({3, 2, 1}))

    # Equality across mutable/frozen set types
    assert {1, 2} == frozenset({2, 1})

    # Canonical undirected edge
    assert frozenset(("A", "B")) == frozenset(("B", "A"))

    # Frozen multiset preserves counts
    assert multiset_key([1, 1, 2]) == multiset_key([2, 1, 1])
    assert multiset_key([1, 1, 2]) != multiset_key([1, 2, 2])

    # Recursive freezing preserves list order
    assert freeze([1, 2]) != freeze([2, 1])

    # Recursive freezing ignores dictionary insertion order
    assert freeze({"a": 1, "b": 2}) == freeze({"b": 2, "a": 1})

    # Powerset size
    assert len(powerset({1, 2, 3, 4})) == 16

    # Query-key semantics
    assert query_key({"a": 1, "b": 2}, ["x", "y"]) == \
           query_key({"b": 2, "a": 1}, ["x", "y"])

    print("All frozenset self-tests passed.")

run_self_test()

All frozenset self-tests passed.
